In [1]:
!git clone https://github.com/romova/SCOPUS_API.git

fatal: destination path 'SCOPUS_API' already exists and is not an empty directory.


In [10]:
import json
import os
import pandas as pd
from collections import defaultdict

def load_coordinates(csv_path):
    coords_df = pd.read_csv(csv_path)
    coords_df = coords_df.dropna(subset=["city", "latitude", "longitude", "country"])
    coords_df = coords_df.drop_duplicates(subset="city", keep="first")

    # Rename 'country' column to 'state' as per user clarification
    coords_df.rename(columns={"country": "state"}, inplace=True)

    return coords_df.set_index("city")[["latitude", "longitude", "state"]].to_dict(orient="index")


def extract_affiliations(affil):
    results = []
    if isinstance(affil, list):
        for a in affil:
            if isinstance(a, dict) and "affiliation-city" in a:
                results.append(a["affiliation-city"])
    elif isinstance(affil, dict):
        city = affil.get("affiliation-city")
        if city:
            results.append(city)
    return results

def extract_authors(authors):
    return [a for a in authors if isinstance(a, str) and a.strip()]

def transform_json(json_files, coords_csv_path, output_path):
    coords_lookup = load_coordinates(coords_csv_path)

    city_year_data = defaultdict(lambda: defaultdict(lambda: {
        "authors": defaultdict(lambda: {
            "been_referenced_by_UWB": 0,
            "cited_UWB": 0
        }),
        "been_referenced_by_UWB": 0,
        "cited_UWB": 0
    }))

    for json_file in json_files:
        with open(json_file, "r", encoding="utf-8") as f:
            data = json.load(f)

        for article in data:
            if article.get("citedby-count", 0) < 10:
                continue  # ✅ Skip articles with less than 5 citations

            year = article.get("year")
            if not year:
                continue

            authors = extract_authors(article.get("authors", []))
            if not authors:
                continue

            # Cited by others → UWB was cited
            citedby_articles = article.get("citedby_articles", [])
            if isinstance(citedby_articles, list):
                for citation in citedby_articles:
                    if isinstance(citation, dict):
                        affils = extract_affiliations(citation.get("affiliation", {}))
                        for city in affils:
                            city = city.strip()
                            if city in coords_lookup:
                                for author in authors:
                                    city_year_data[city][year]["authors"][author]["cited_UWB"] += 1
                                city_year_data[city][year]["cited_UWB"] += 1

            # UWB referenced others → they were referenced by UWB
            references = article.get("references", [])
            if isinstance(references, list):
                for reference in references:
                    if isinstance(reference, dict):
                        affils = extract_affiliations(reference.get("affiliation", {}))
                        for city in affils:
                            city = city.strip()
                            if city in coords_lookup:
                                for author in authors:
                                    city_year_data[city][year]["authors"][author]["been_referenced_by_UWB"] += 1
                                city_year_data[city][year]["been_referenced_by_UWB"] += 1

    # Build final structure
    final_output = []
    for city, years_data in city_year_data.items():
        coords = coords_lookup.get(city)
        if not coords:
            continue

        city_entry = {
            "city": city,
            "state": coords["state"],  # from 'country' column
            "latitude": coords["latitude"],
            "longitude": coords["longitude"],
            "years": []
        }

        for year, stats in years_data.items():
            authors_list = []
            for author, counts in stats["authors"].items():
                authors_list.append({
                    "name": author,
                    "been_referenced_by_UWB": counts["been_referenced_by_UWB"],
                    "cited_UWB": counts["cited_UWB"]
                })

            city_entry["years"].append({
                "year": year,
                "authors": authors_list,
                "been_referenced_by_UWB": stats["been_referenced_by_UWB"],
                "cited_UWB": stats["cited_UWB"]
            })

        if city_entry["years"]:
            final_output.append(city_entry)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(final_output, f, ensure_ascii=False, indent=2)

    print(f"✅ Transformed data saved to: {output_path}")


In [11]:
json_files = ['SCOPUS_API/data_by_year/all_articles_by_institution_cited_1992.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_1993.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_1994.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_1995.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_1996.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_1997.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_1998.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_1999.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2000.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2001.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2002.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2003.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2004.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2005.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2006.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2007.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2008.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2009.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2010.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2011.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2012.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2013.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2014.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2015.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2016.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2017.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2018.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2019.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2020.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2021.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2022.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2023.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2024.json',
              'SCOPUS_API/data_by_year/all_articles_by_institution_cited_2025.json']

transform_json(json_files, "SCOPUS_API/institutions_with_coordinates.csv", "final_transformed_output.json")

✅ Transformed data saved to: final_transformed_output.json


In [12]:
import json

def merge_plzen_into_pilsen(input_path, output_path):
    with open(input_path, "r", encoding="utf-8") as f:
        data = json.load(f)

    pilsen_entry = None
    new_data = []

    for entry in data:
        city_name = entry["city"].strip()
        if city_name.lower() == "pilsen":
            pilsen_entry = entry
        elif city_name.lower() == "plzen":
            if pilsen_entry is None:
                pilsen_entry = {
                    "city": "Pilsen",
                    "state": entry.get("state", ""),
                    "latitude": entry.get("latitude"),
                    "longitude": entry.get("longitude"),
                    "years": []
                }

            # Merge years
            existing_years = {y["year"]: y for y in pilsen_entry["years"]}
            for year_info in entry["years"]:
                y = year_info["year"]
                if y in existing_years:
                    # Merge authors
                    existing_authors = {a["name"]: a for a in existing_years[y]["authors"]}
                    for author in year_info["authors"]:
                        name = author["name"]
                        if name in existing_authors:
                            existing_authors[name]["been_referenced_by_UWB"] += author["been_referenced_by_UWB"]
                            existing_authors[name]["cited_UWB"] += author["cited_UWB"]
                        else:
                            existing_years[y]["authors"].append(author)

                    # Merge city-year totals
                    existing_years[y]["been_referenced_by_UWB"] += year_info["been_referenced_by_UWB"]
                    existing_years[y]["cited_UWB"] += year_info["cited_UWB"]
                else:
                    pilsen_entry["years"].append(year_info)
        else:
            new_data.append(entry)

    if pilsen_entry:
        new_data.append(pilsen_entry)

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(new_data, f, ensure_ascii=False, indent=2)

    print(f"✅ 'Plzen' merged into 'Pilsen' and saved to: {output_path}")

In [13]:
merge_plzen_into_pilsen("final_transformed_output.json", "merged_output.json")

✅ 'Plzen' merged into 'Pilsen' and saved to: merged_output.json


In [14]:
file_path = 'merged_output.json'

In [15]:
import os
from google.colab import files

# Print file size
if os.path.exists(file_path):
    size_bytes = os.path.getsize(file_path)
    size_mb = size_bytes / (1024 * 1024)
    print(f"📦 File size: {size_mb:.2f} MB")
else:
    print("❌ File not found.")

📦 File size: 625.96 MB


In [16]:
# Download the file
if os.path.exists(file_path):
    files.download(file_path)
else:
    print("❌ File not found.")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>